In [0]:
bronze_customers_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
    "bronze/customers"
)

bronze_customers_df =(
    spark.read.format("delta")
    .load(bronze_customers_path)
)
display(bronze_customers_df)

customer_id,first_name,last_name,email,city,state,country,customer_segment,created_date,updated_date
1,Arjun,Sharma,arjun.sharma1@example.com,Indore,Madhya Pradesh,India,Consumer,2024-10-08,2024-12-09
2,Rahul,Nair,rahul.nair2@example.com,Mumbai,Maharashtra,India,Premium,2025-07-12,2025-08-03
3,Pooja,Sharma,pooja.sharma3@example.com,Pune,Maharashtra,India,Consumer,2024-04-05,2024-05-30
4,Nisha,Khan,nisha.khan4@example.com,Pune,Maharashtra,India,Premium,2025-07-28,2025-09-16
5,Riya,Kulkarni,riya.kulkarni5@example.com,Delhi,Delhi,India,Corporate,2025-04-04,2025-09-01
6,Aarav,Reddy,aarav.reddy6@example.com,Lucknow,Uttar Pradesh,India,Corporate,2024-06-12,2024-12-07
7,Ananya,Joshi,ananya.joshi7@example.com,Nagpur,Maharashtra,India,Consumer,2024-08-08,2024-11-02
8,Aditya,Kulkarni,aditya.kulkarni8@example.com,Mumbai,Maharashtra,India,Premium,2025-01-02,2025-03-31
9,Neha,Reddy,neha.reddy9@example.com,Pune,Maharashtra,India,Consumer,2025-04-15,2025-08-30
10,Sneha,Patil,sneha.patil10@example.com,Kolkata,West Bengal,India,Premium,2024-10-27,2025-04-05


In [0]:
bronze_customers_df.printSchema()

root
 |-- customer_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- created_date: date (nullable = true)
 |-- updated_date: date (nullable = true)



In [0]:
print("Bronze customer count:", bronze_customers_df.count())

Bronze customer count: 100


In [0]:
display(
    bronze_customers_df.select(
        "customer_id",
        "first_name",
        "last_name",
        "email",
        "city",
        "state",
        "country",
        "customer_segment",
        "created_date",
        "updated_date"
    )
)

customer_id,first_name,last_name,email,city,state,country,customer_segment,created_date,updated_date
1,Arjun,Sharma,arjun.sharma1@example.com,Indore,Madhya Pradesh,India,Consumer,2024-10-08,2024-12-09
2,Rahul,Nair,rahul.nair2@example.com,Mumbai,Maharashtra,India,Premium,2025-07-12,2025-08-03
3,Pooja,Sharma,pooja.sharma3@example.com,Pune,Maharashtra,India,Consumer,2024-04-05,2024-05-30
4,Nisha,Khan,nisha.khan4@example.com,Pune,Maharashtra,India,Premium,2025-07-28,2025-09-16
5,Riya,Kulkarni,riya.kulkarni5@example.com,Delhi,Delhi,India,Corporate,2025-04-04,2025-09-01
6,Aarav,Reddy,aarav.reddy6@example.com,Lucknow,Uttar Pradesh,India,Corporate,2024-06-12,2024-12-07
7,Ananya,Joshi,ananya.joshi7@example.com,Nagpur,Maharashtra,India,Consumer,2024-08-08,2024-11-02
8,Aditya,Kulkarni,aditya.kulkarni8@example.com,Mumbai,Maharashtra,India,Premium,2025-01-02,2025-03-31
9,Neha,Reddy,neha.reddy9@example.com,Pune,Maharashtra,India,Consumer,2025-04-15,2025-08-30
10,Sneha,Patil,sneha.patil10@example.com,Kolkata,West Bengal,India,Premium,2024-10-27,2025-04-05


In [0]:
from pyspark.sql.functions import col, sum, when

null_check = bronze_customers_df.select(
    *[
        sum(
            when(col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in bronze_customers_df.columns
    ]
)

display(null_check)

customer_id,first_name,last_name,email,city,state,country,customer_segment,created_date,updated_date
0,0,0,0,0,0,0,0,0,0


In [0]:
print(
    "Duplicate customer IDs:",
    bronze_customers_df.count()
    - bronze_customers_df.select("customer_id").distinct().count()
)

Duplicate customer IDs: 0


In [0]:
print("Customer Segments:")
display(
    bronze_customers_df
    .select("customer_segment")
    .distinct()
    .orderBy("customer_segment")
)

print("Countries:")
display(
    bronze_customers_df
    .select("country")
    .distinct()
    .orderBy("country")
)

print("States:")
display(
    bronze_customers_df
    .select("state")
    .distinct()
    .orderBy("state")
)

Customer Segments:


customer_segment
Consumer
Corporate
Premium


Countries:


country
India
UAE


States:


state
Delhi
Dubai
Gujarat
Karnataka
Madhya Pradesh
Maharashtra
Odisha
Rajasthan
Tamil Nadu
Telangana


In [0]:
from pyspark.sql.functions import (
    col,
    trim,
    lower,
    when
)

silver_customers_df = (
    bronze_customers_df

    # Standardize string fields
    .withColumn("first_name", trim(col("first_name")))
    .withColumn("last_name", trim(col("last_name")))
    .withColumn("email", lower(trim(col("email"))))
    .withColumn("city", trim(col("city")))
    .withColumn("state", trim(col("state")))
    .withColumn("country", trim(col("country")))

    # Validate customer segment
    .withColumn(
        "customer_segment",
        when(
            col("customer_segment").isin(
                "Consumer",
                "Corporate",
                "Premium"
            ),
            col("customer_segment")
        ).otherwise(None)
    )

    # Remove duplicate customer records
    .dropDuplicates(["customer_id"])
)

In [0]:
display(silver_customers_df)

customer_id,first_name,last_name,email,city,state,country,customer_segment,created_date,updated_date
1,Arjun,Sharma,arjun.sharma1@example.com,Indore,Madhya Pradesh,India,Consumer,2024-10-08,2024-12-09
2,Rahul,Nair,rahul.nair2@example.com,Mumbai,Maharashtra,India,Premium,2025-07-12,2025-08-03
3,Pooja,Sharma,pooja.sharma3@example.com,Pune,Maharashtra,India,Consumer,2024-04-05,2024-05-30
4,Nisha,Khan,nisha.khan4@example.com,Pune,Maharashtra,India,Premium,2025-07-28,2025-09-16
5,Riya,Kulkarni,riya.kulkarni5@example.com,Delhi,Delhi,India,Corporate,2025-04-04,2025-09-01
6,Aarav,Reddy,aarav.reddy6@example.com,Lucknow,Uttar Pradesh,India,Corporate,2024-06-12,2024-12-07
7,Ananya,Joshi,ananya.joshi7@example.com,Nagpur,Maharashtra,India,Consumer,2024-08-08,2024-11-02
8,Aditya,Kulkarni,aditya.kulkarni8@example.com,Mumbai,Maharashtra,India,Premium,2025-01-02,2025-03-31
9,Neha,Reddy,neha.reddy9@example.com,Pune,Maharashtra,India,Consumer,2025-04-15,2025-08-30
10,Sneha,Patil,sneha.patil10@example.com,Kolkata,West Bengal,India,Premium,2024-10-27,2025-04-05


In [0]:
print("Silver customer count:", silver_customers_df.count())

Silver customer count: 100


In [0]:
from pyspark.sql.functions import col, sum, when

print("Silver customer count:", silver_customers_df.count())

print(
    "Distinct customer IDs:",
    silver_customers_df.select("customer_id").distinct().count()
)

print(
    "Duplicate customer IDs:",
    silver_customers_df.count()
    - silver_customers_df.select("customer_id").distinct().count()
)

Silver customer count: 100
Distinct customer IDs: 100
Duplicate customer IDs: 0


In [0]:
invalid_segments = silver_customers_df.filter(
    ~col("customer_segment").isin(
        "Consumer",
        "Corporate",
        "Premium"
    )
)

print("Invalid customer segments:", invalid_segments.count())

Invalid customer segments: 0


In [0]:
silver_null_check = silver_customers_df.select(
    *[
        sum(
            when(col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in silver_customers_df.columns
    ]
)

display(silver_null_check)

customer_id,first_name,last_name,email,city,state,country,customer_segment,created_date,updated_date
0,0,0,0,0,0,0,0,0,0


In [0]:
silver_customers_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
    "silver/customers"
)

(
    silver_customers_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_customers_path)
)

print("Silver Customers written successfully.")

Silver Customers written successfully.


In [0]:
silver_customers_check = (
    spark.read
    .format("delta")
    .load(silver_customers_path)
)

print("Silver Customers records:", silver_customers_check.count())

display(silver_customers_check)

Silver Customers records: 100


customer_id,first_name,last_name,email,city,state,country,customer_segment,created_date,updated_date
12,Akash,Reddy,akash.reddy12@example.com,Jaipur,Rajasthan,India,Consumer,2024-03-22,2024-05-20
18,Vivaan,Reddy,vivaan.reddy18@example.com,Chennai,Tamil Nadu,India,Consumer,2025-02-14,2025-04-23
38,Priya,Kulkarni,priya.kulkarni38@example.com,Ahmedabad,Gujarat,India,Premium,2025-01-17,2025-05-09
67,Isha,Joshi,isha.joshi67@example.com,Nagpur,Maharashtra,India,Premium,2025-03-24,2025-08-10
70,Arjun,Pawar,arjun.pawar70@example.com,Mumbai,Maharashtra,India,Corporate,2025-07-20,2025-08-28
93,Simran,Gupta,simran.gupta93@example.com,Ahmedabad,Gujarat,India,Premium,2024-05-07,2024-07-22
16,Akash,Singh,akash.singh16@example.com,Hyderabad,Telangana,India,Premium,2025-01-23,2025-04-02
64,Priya,Deshmukh,priya.deshmukh64@example.com,Chennai,Tamil Nadu,India,Corporate,2024-03-15,2024-03-17
74,Rahul,Shah,rahul.shah74@example.com,Jaipur,Rajasthan,India,Premium,2024-06-14,2024-10-05
94,Priya,Shah,priya.shah94@example.com,Bengaluru,Karnataka,India,Premium,2024-11-30,2025-03-13


In [0]:
bronze_products_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
    "bronze/products"
)

bronze_products_df = (
    spark.read
    .format("delta")
    .load(bronze_products_path)
)

display(bronze_products_df)

product_id,product_name,category,subcategory,brand,unit_price,cost_price,created_date,updated_date
1001,Apex Mobile 1001,Electronics,Mobile,Zenith,37496,27030.64,2024-01-01,2025-01-01
1002,Nova Mobile 1002,Electronics,Mobile,Vertex,44233,32363.29,2024-01-01,2025-01-01
1003,Vertex Mobile 1003,Electronics,Mobile,Vertex,29744,16555.36,2024-01-01,2025-01-01
1004,Prime Laptop 1004,Electronics,Laptop,Apex,32392,25247.27,2024-01-01,2025-01-01
1005,Prime Laptop 1005,Electronics,Laptop,Prime,9845,7770.23,2024-01-01,2025-01-01
1006,Vertex Laptop 1006,Electronics,Laptop,Zenith,75754,55907.64,2024-01-01,2025-01-01
1007,Prime Audio 1007,Electronics,Audio,Vertex,65099,38398.27,2024-01-01,2025-01-01
1008,Nova Audio 1008,Electronics,Audio,Prime,1026,624.92,2024-01-01,2025-01-01
1009,Prime Audio 1009,Electronics,Audio,Nova,68189,47766.36,2024-01-01,2025-01-01
1010,Prime Accessories 1010,Electronics,Accessories,Vertex,16206,12419.99,2024-01-01,2025-01-01


In [0]:
bronze_products_df.printSchema()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- cost_price: double (nullable = true)
 |-- created_date: date (nullable = true)
 |-- updated_date: date (nullable = true)



In [0]:
from pyspark.sql.functions import col, sum, when

print("Categories:")
display(
    bronze_products_df
    .select("category")
    .distinct()
    .orderBy("category")
)

print("Subcategories:")
display(
    bronze_products_df
    .select("subcategory")
    .distinct()
    .orderBy("subcategory")
)

print("Brands:")
display(
    bronze_products_df
    .select("brand")
    .distinct()
    .orderBy("brand")
)

print("Duplicate product IDs:")
print(
    bronze_products_df.count()
    - bronze_products_df.select("product_id").distinct().count()
)

print("NULL check:")
display(
    bronze_products_df.select(
        *[
            sum(
                when(col(c).isNull(), 1).otherwise(0)
            ).alias(c)
            for c in bronze_products_df.columns
        ]
    )
)

Categories:


category
Electronics
Fashion
Grocery
Home
Sports


Subcategories:


subcategory
Accessories
Appliances
Audio
Beverages
Cricket
Decor
Fitness
Footwear
Furniture
Kitchen


Brands:


brand
Apex
Nova
Prime
UrbanEdge
Vertex
Zenith


Duplicate product IDs:
0
NULL check:


product_id,product_name,category,subcategory,brand,unit_price,cost_price,created_date,updated_date
0,0,0,0,0,0,0,0,0


In [0]:
print(
    "Invalid prices:",
    bronze_products_df.filter(
        (col("unit_price") <= 0) |
        (col("cost_price") <= 0) |
        (col("cost_price") > col("unit_price"))
    ).count()
)

Invalid prices: 0


In [0]:
from pyspark.sql.functions import col, trim

silver_products_df = (
    bronze_products_df

    # Standardize text fields
    .withColumn("product_name", trim(col("product_name")))
    .withColumn("category", trim(col("category")))
    .withColumn("subcategory", trim(col("subcategory")))
    .withColumn("brand", trim(col("brand")))

    # Remove duplicate product records
    .dropDuplicates(["product_id"])
)

In [0]:
invalid_products_df = silver_products_df.filter(
    (col("unit_price") <= 0) |
    (col("cost_price") <= 0) |
    (col("cost_price") > col("unit_price"))
)

print("Invalid products:", invalid_products_df.count())

Invalid products: 0


In [0]:
print("Silver product count:", silver_products_df.count())

print(
    "Distinct product IDs:",
    silver_products_df.select("product_id").distinct().count()
)

Silver product count: 60
Distinct product IDs: 60


In [0]:
display(silver_products_df)

product_id,product_name,category,subcategory,brand,unit_price,cost_price,created_date,updated_date
1001,Apex Mobile 1001,Electronics,Mobile,Zenith,37496,27030.64,2024-01-01,2025-01-01
1002,Nova Mobile 1002,Electronics,Mobile,Vertex,44233,32363.29,2024-01-01,2025-01-01
1003,Vertex Mobile 1003,Electronics,Mobile,Vertex,29744,16555.36,2024-01-01,2025-01-01
1004,Prime Laptop 1004,Electronics,Laptop,Apex,32392,25247.27,2024-01-01,2025-01-01
1005,Prime Laptop 1005,Electronics,Laptop,Prime,9845,7770.23,2024-01-01,2025-01-01
1006,Vertex Laptop 1006,Electronics,Laptop,Zenith,75754,55907.64,2024-01-01,2025-01-01
1007,Prime Audio 1007,Electronics,Audio,Vertex,65099,38398.27,2024-01-01,2025-01-01
1008,Nova Audio 1008,Electronics,Audio,Prime,1026,624.92,2024-01-01,2025-01-01
1009,Prime Audio 1009,Electronics,Audio,Nova,68189,47766.36,2024-01-01,2025-01-01
1010,Prime Accessories 1010,Electronics,Accessories,Vertex,16206,12419.99,2024-01-01,2025-01-01


In [0]:
print("Silver product count:", silver_products_df.count())

print(
    "Distinct product IDs:",
    silver_products_df.select("product_id").distinct().count()
)

print(
    "Duplicate product IDs:",
    silver_products_df.count()
    - silver_products_df.select("product_id").distinct().count()
)

print(
    "Invalid prices:",
    silver_products_df.filter(
        (col("unit_price") <= 0) |
        (col("cost_price") <= 0) |
        (col("cost_price") > col("unit_price"))
    ).count()
)

Silver product count: 60
Distinct product IDs: 60
Duplicate product IDs: 0
Invalid prices: 0


In [0]:
silver_products_null_check = silver_products_df.select(
    *[
        sum(
            when(col(c).isNull(), 1).otherwise(0)
        ).alias(c)
        for c in silver_products_df.columns
    ]
)

display(silver_products_null_check)

product_id,product_name,category,subcategory,brand,unit_price,cost_price,created_date,updated_date
0,0,0,0,0,0,0,0,0


In [0]:
silver_products_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
    "silver/products"
)

(
    silver_products_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_products_path)
)

print("Silver Products written successfully.")

Silver Products written successfully.


In [0]:
silver_products_check = (
    spark.read
    .format("delta")
    .load(silver_products_path)
)

print("Persisted Silver Products:", silver_products_check.count())

Persisted Silver Products: 60


In [0]:
bronze_stores_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
    "bronze/stores"
)

bronze_regions_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
    "bronze/regions"
)

bronze_stores_df = (
    spark.read
    .format("delta")
    .load(bronze_stores_path)
)

bronze_regions_df = (
    spark.read
    .format("delta")
    .load(bronze_regions_path)
)

display(bronze_stores_df)
display(bronze_regions_df)

store_id,store_name,city,state,country,region_id
101,GlobalMart Pune,Pune,Maharashtra,India,3
102,GlobalMart Mumbai,Mumbai,Maharashtra,India,3
103,GlobalMart Nagpur,Nagpur,Maharashtra,India,5
104,GlobalMart Delhi,Delhi,Delhi,India,1
105,GlobalMart Jaipur,Jaipur,Rajasthan,India,1
106,GlobalMart Chennai,Chennai,Tamil Nadu,India,2
107,GlobalMart Bengaluru,Bengaluru,Karnataka,India,2
108,GlobalMart Hyderabad,Hyderabad,Telangana,India,2
109,GlobalMart Kolkata,Kolkata,West Bengal,India,4
110,GlobalMart Bhubaneswar,Bhubaneswar,Odisha,India,4


region_id,region_name,country
1,North,India
2,South,India
3,West,India
4,East,India
5,Central,India
6,International,UAE


In [0]:
bronze_stores_df.printSchema()
bronze_regions_df.printSchema()

root
 |-- store_id: integer (nullable = true)
 |-- store_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- region_id: integer (nullable = true)

root
 |-- region_id: integer (nullable = true)
 |-- region_name: string (nullable = true)
 |-- country: string (nullable = true)



In [0]:
print("Stores:", bronze_stores_df.count())
print("Regions:", bronze_regions_df.count())

print("Store IDs:")
display(
    bronze_stores_df
    .select("store_id")
    .distinct()
    .orderBy("store_id")
)

print("Regions:")
display(
    bronze_regions_df
    .orderBy("region_id")
)

Stores: 15
Regions: 6
Store IDs:


store_id
101
102
103
104
105
106
107
108
109
110


Regions:


region_id,region_name,country
1,North,India
2,South,India
3,West,India
4,East,India
5,Central,India
6,International,UAE


In [0]:
print(
    "Duplicate store IDs:",
    bronze_stores_df.count()
    - bronze_stores_df.select("store_id").distinct().count()
)

print(
    "Duplicate region IDs:",
    bronze_regions_df.count()
    - bronze_regions_df.select("region_id").distinct().count()
)

Duplicate store IDs: 0
Duplicate region IDs: 0


In [0]:
invalid_store_regions = (
    bronze_stores_df
    .join(
        bronze_regions_df.select("region_id"),
        on="region_id",
        how="left_anti"
    )
)

print(
    "Stores with invalid region_id:",
    invalid_store_regions.count()
)

display(invalid_store_regions)

Stores with invalid region_id: 0


region_id,store_id,store_name,city,state,country


In [0]:
region_country_mismatch = (
    bronze_stores_df.alias("s")
    .join(
        bronze_regions_df.alias("r"),
        col("s.region_id") == col("r.region_id"),
        "inner"
    )
    .filter(
        col("s.country") != col("r.country")
    )
    .select(
        col("s.store_id"),
        col("s.country").alias("store_country"),
        col("s.region_id"),
        col("r.region_name"),
        col("r.country").alias("region_country")
    )
)

print(
    "Store/region country mismatches:",
    region_country_mismatch.count()
)

display(region_country_mismatch)

Store/region country mismatches: 0


store_id,store_country,region_id,region_name,region_country


In [0]:
from pyspark.sql.functions import col, trim

silver_stores_df = (
    bronze_stores_df
    .withColumn("store_name", trim(col("store_name")))
    .withColumn("city", trim(col("city")))
    .withColumn("state", trim(col("state")))
    .withColumn("country", trim(col("country")))
    .dropDuplicates(["store_id"])
)

In [0]:
silver_regions_df = (
    bronze_regions_df
    .withColumn("region_name", trim(col("region_name")))
    .withColumn("country", trim(col("country")))
    .dropDuplicates(["region_id"])
)

In [0]:
print("Silver Stores:", silver_stores_df.count())
print("Silver Regions:", silver_regions_df.count())

print(
    "Distinct Store IDs:",
    silver_stores_df.select("store_id").distinct().count()
)

print(
    "Distinct Region IDs:",
    silver_regions_df.select("region_id").distinct().count()
)

Silver Stores: 15
Silver Regions: 6
Distinct Store IDs: 15
Distinct Region IDs: 6


In [0]:
silver_stores_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
    "silver/stores"
)

silver_regions_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
    "silver/regions"
)

(
    silver_stores_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_stores_path)
)

(
    silver_regions_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_regions_path)
)

print("Silver Stores and Regions written successfully.")

Silver Stores and Regions written successfully.


In [0]:
bronze_orders_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
    "bronze/orders"
)

bronze_orders_df = (
    spark.read
    .format("delta")
    .load(bronze_orders_path)
)

display(bronze_orders_df)

order_id,customer_id,store_id,order_date,order_status,sales_channel,payment_id,updated_date
5001,98,104,2025-09-08T17:00:00.000Z,Cancelled,Mobile App,90001,2025-09-11T17:00:00.000Z
5002,12,105,2025-08-18T00:00:00.000Z,Completed,Retail,90002,2025-08-19T00:00:00.000Z
5003,75,106,2025-06-06T21:00:00.000Z,Cancelled,Mobile App,90003,2025-06-08T21:00:00.000Z
5004,71,106,2025-08-06T23:00:00.000Z,Completed,Mobile App,90004,2025-08-09T23:00:00.000Z
5005,33,104,2025-05-19T09:00:00.000Z,Completed,Mobile App,90005,2025-05-20T09:00:00.000Z
5006,96,109,2025-06-11T03:00:00.000Z,Completed,Online,90006,2025-06-12T03:00:00.000Z
5007,93,110,2025-09-05T08:00:00.000Z,Pending,Mobile App,90007,2025-09-07T08:00:00.000Z
5008,38,104,2025-02-21T06:00:00.000Z,Completed,Online,90008,2025-02-23T06:00:00.000Z
5009,69,103,2025-01-08T22:00:00.000Z,Completed,Online,90009,2025-01-08T22:00:00.000Z
5010,90,103,2025-10-11T09:00:00.000Z,Cancelled,Online,90010,2025-10-11T09:00:00.000Z


In [0]:
bronze_orders_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- store_id: integer (nullable = true)
 |-- order_date: timestamp (nullable = true)
 |-- order_status: string (nullable = true)
 |-- sales_channel: string (nullable = true)
 |-- payment_id: integer (nullable = true)
 |-- updated_date: timestamp (nullable = true)



In [0]:
from pyspark.sql.functions import col, trim

silver_orders_df = (
    bronze_orders_df
    .withColumn("order_status", trim(col("order_status")))
    .withColumn("sales_channel", trim(col("sales_channel")))
    .dropDuplicates(["order_id"])
)

silver_orders_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
    "silver/orders"
)

(
    silver_orders_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_orders_path)
)

print("Silver Orders written successfully.")
print("Silver Orders count:", silver_orders_df.count())

Silver Orders written successfully.
Silver Orders count: 500


In [0]:
print(
    "Distinct order IDs:",
    silver_orders_df.select("order_id").distinct().count()
)

print(
    "Duplicate order IDs:",
    silver_orders_df.count()
    - silver_orders_df.select("order_id").distinct().count()
)

Distinct order IDs: 500
Duplicate order IDs: 0


In [0]:
invalid_customers = (
    silver_orders_df
    .join(
        silver_customers_df.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
)

invalid_stores = (
    silver_orders_df
    .join(
        silver_stores_df.select("store_id"),
        on="store_id",
        how="left_anti"
    )
)

print("Orders with invalid customer_id:", invalid_customers.count())
print("Orders with invalid store_id:", invalid_stores.count())

Orders with invalid customer_id: 0
Orders with invalid store_id: 0


In [0]:
bronze_order_items_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
    "bronze/order_items"
)

bronze_order_items_df = (
    spark.read
    .format("delta")
    .load(bronze_order_items_path)
)

bronze_order_items_df.printSchema()
print("Bronze Order Items:", bronze_order_items_df.count())

root
 |-- order_item_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- updated_date: timestamp (nullable = true)

Bronze Order Items: 1298


In [0]:
from pyspark.sql.functions import col, trim

silver_order_items_df = (
    bronze_order_items_df
    .dropDuplicates(["order_item_id"])
)

silver_order_items_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
    "silver/order_items"
)

(
    silver_order_items_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_order_items_path)
)

print("Silver Order Items written successfully.")
print("Silver Order Items:", silver_order_items_df.count())

Silver Order Items written successfully.
Silver Order Items: 1298


In [0]:
print(
    "Distinct order_item IDs:",
    silver_order_items_df.select("order_item_id").distinct().count()
)

print(
    "Duplicate order_item IDs:",
    silver_order_items_df.count()
    - silver_order_items_df.select("order_item_id").distinct().count()
)

Distinct order_item IDs: 1298
Duplicate order_item IDs: 0


In [0]:
invalid_orders = (
    silver_order_items_df
    .join(
        silver_orders_df.select("order_id"),
        on="order_id",
        how="left_anti"
    )
)

invalid_products = (
    silver_order_items_df
    .join(
        silver_products_df.select("product_id"),
        on="product_id",
        how="left_anti"
    )
)

print("Order Items with invalid order_id:", invalid_orders.count())
print("Order Items with invalid product_id:", invalid_products.count())

Order Items with invalid order_id: 0
Order Items with invalid product_id: 0


In [0]:
bronze_payments_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
    "bronze/payments"
)

bronze_payments_df = (
    spark.read
    .format("delta")
    .load(bronze_payments_path)
)

bronze_payments_df.printSchema()
print("Bronze Payments:", bronze_payments_df.count())

root
 |-- payment_id: integer (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- payment_status: string (nullable = true)
 |-- payment_amount: double (nullable = true)
 |-- payment_date: timestamp (nullable = true)

Bronze Payments: 500


In [0]:
from pyspark.sql.functions import col, trim

silver_payments_df = (
    bronze_payments_df
    .withColumn("payment_method", trim(col("payment_method")))
    .withColumn("payment_status", trim(col("payment_status")))
    .dropDuplicates(["payment_id"])
)

silver_payments_path = (
    "abfss://globalmart@globalmartpiyush.dfs.core.windows.net/"
    "silver/payments"
)

(
    silver_payments_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_payments_path)
)

print("Silver Payments written successfully.")
print("Silver Payments:", silver_payments_df.count())

Silver Payments written successfully.
Silver Payments: 500


In [0]:
print(
    "Distinct payment IDs:",
    silver_payments_df.select("payment_id").distinct().count()
)

print(
    "Duplicate payment IDs:",
    silver_payments_df.count()
    - silver_payments_df.select("payment_id").distinct().count()
)

Distinct payment IDs: 500
Duplicate payment IDs: 0


In [0]:
invalid_orders = (
    silver_payments_df
    .join(
        silver_orders_df.select("order_id"),
        on="order_id",
        how="left_anti"
    )
)

print(
    "Payments with invalid order_id:",
    invalid_orders.count()
)

Payments with invalid order_id: 0
